# Multimodal AI — Blood Work Analysis

In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage
from langchain.tools import tool
from langchain.agents import create_agent
import base64  #encoding image  - kind of string representation

load_dotenv()

True

## Encode the image and send to the vision model

In [2]:
from groq import Groq

client = Groq()

with open("blood_work.png", "rb") as f:
    image_b64 = base64.b64encode(f.read()).decode("utf-8")

completion = client.chat.completions.create(
    model="qwen/qwen3.8-27b",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}
                },
                {
                    "type": "text", "text": "This is a blood work report. Extract all test results and flag any values outside the normal range."
                }
            ]
        }
    ],
    temperature=0.6,
    max_completion_tokens=2048,
    top_p=0.95,
    stream=True
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")

Based on the blood work report provided for **Rajesh Sharma (Age 48, Male)**, here is the extraction of all test results with flags for values outside the normal range.

### **COMPLETE BLOOD COUNT (CBC)**
*   **Hemoglobin:** 15.1 g/dL *(Normal: 13.5–17.5)* — **Normal**
*   **Hematocrit:** 44% *(Normal: 41–53%)* — **Normal**
*   **WBC:** 6.8 x 10³/uL *(Normal: 4.5–11.0)* — **Normal**
*   **Platelets:** 220 x 10³/uL *(Normal: 150–400)* — **Normal**

---

### **LIPID PANEL**
*   **Total Cholesterol:** 238 mg/dL *(Normal: <200)* — **⚠️ HIGH**
*   **LDL Cholesterol:** 162 mg/dL *(Normal: <100)* — **⚠️ HIGH**
*   **HDL Cholesterol:** 36 mg/dL *(Normal: >40)* — **⚠️ LOW**
*   **Triglycerides:** 188 mg/dL *(Normal: <150)* — **⚠️ HIGH**

---

### **METABOLIC PANEL**
*   **Glucose (Fasting):** 92 mg/dL *(Normal: 70–99)* — **Normal**
*   **HbA1c:** 5.3% *(Normal: <5.7%)* — **Normal**
*   **Creatinine:** 1.0 mg/dL *(Normal: 0.7–1.3)* — **Normal**
*   **eGFR:** 82 mL/min *(Normal: >60)* — **Normal*

## Suggest Diet Plan Agent

The agent reads the blood work image, categorises the condition, then calls the diet tool.

In [3]:
@tool
def get_diet_recommendation(condition: str) -> dict:
    """Given a health condition, returns a diet plan. Condition must be one of: normal, high_cholesterol, high_sugar."""
    diet_plans = {
        "high_cholesterol": {
            "eat":        ["fruits", "vegetables", "whole grains", "lean protein"],
            "do_not_eat": ["red meat", "fried food", "full-fat dairy", "processed snacks"],
        },
        "high_sugar": {
            "eat":        ["vegetables", "whole grains", "legumes", "nuts"],
            "do_not_eat": ["white rice", "white sugar", "junk food", "sugary drinks"],
        },
        "normal": {
            "eat":        ["vegetables", "fruits", "whole grains", "lean protein"],
            "do_not_eat": ["excessive sugar", "processed food", "trans fats"],
        },
    }
    return diet_plans.get(condition, diet_plans["normal"])

In [8]:
SYSTEM_PROMPT = """
You are a helpful medical and nutrition assistant.
For the input blood work image, extract the numbers and the normal range, then categorize
the condition as one of: normal, high_cholesterol, high_sugar.
Then call the appropriate tool to retrieve and present the diet plan.
"""

diet_agent = create_agent(
    ChatGroq(model='qwen/qwen3.8-27b'),
    tools=[get_diet_recommendation],
    system_prompt=SYSTEM_PROMPT,
)

In [9]:
result = diet_agent.invoke({
    "messages": [HumanMessage(content=[
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        {"type": "text",      "text": "Analyse this blood work report and suggest a diet plan."},
    ])]
})

print(result["messages"][-1].content)

Based on your high cholesterol levels, here is your recommended diet plan:

**Recommended Foods to Eat:**
*   Fruits
*   Vegetables
*   Whole grains
*   Lean protein

**Foods to Avoid:**
*   Red meat
*   Fried food
*   Full-fat dairy
*   Processed snacks

Please consult with your healthcare provider or a registered dietitian to create a comprehensive plan tailored to your specific needs.
